# 02 — Silver Transform: Project Profitability & Utilisation Analytics
Enriches Bronze tables with business-logic derived fields.

| Silver Table | Derived Fields |
|---|---|
| silver_consultants | tenure_years, grade_band |
| silver_clients | client_concentration_flag |
| silver_engagements | delivery_health_flag, margin_band, duration_days |
| silver_timesheets | billable_utilisation_rate |

In [ ]:
from pyspark.sql import functions as F


In [ ]:
# ── silver_consultants ───────────────────────────────────────────────────────
df_con = spark.table("bronze_consultants")

df_sc = (
    df_con
    .withColumn("tenure_years",
        F.round(F.datediff(F.current_date(), F.col("hire_date")) / 365.25, 1))
    .withColumn("grade_band",
        F.when(F.col("grade").isin("Analyst", "Consultant"), "Junior")
         .when(F.col("grade").isin("Senior Consultant", "Manager"), "Mid-Level")
         .otherwise("Senior"))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sc.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_consultants")
print(f"silver_consultants: {df_sc.count():,}")

In [ ]:
# ── silver_clients ───────────────────────────────────────────────────────────
df_cli = spark.table("bronze_clients")

# Flag clients where contract value > 10% of total portfolio
total_cv = df_cli.agg(F.sum("contract_value_gbp").alias("t")).collect()[0]["t"]

df_scli = (
    df_cli
    .withColumn("concentration_pct",
        F.round(F.col("contract_value_gbp") / total_cv * 100, 2))
    .withColumn("client_concentration_flag",
        (F.col("concentration_pct") > 10).cast("int"))
    .withColumn("nps_band",
        F.when(F.col("nps_score") >= 50, "Promoter")
         .when(F.col("nps_score") >= 0, "Passive")
         .otherwise("Detractor"))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_scli.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_clients")
print(f"silver_clients: {df_scli.count():,}")

In [ ]:
# ── silver_engagements ───────────────────────────────────────────────────────
df_eng = spark.table("bronze_engagements")

df_seng = (
    df_eng
    .withColumn("duration_days",
        F.datediff(F.col("planned_end_date"), F.col("start_date")))
    .withColumn("margin_band",
        F.when(F.col("margin_pct") >= 20, "High (≥20%)")
         .when(F.col("margin_pct") >= 5,  "Normal (5–20%)")
         .when(F.col("margin_pct") >= 0,  "Low (0–5%)")
         .otherwise("Loss-Making (<0%)"))
    .withColumn("delivery_health_flag",
        (F.col("status").isin("At Risk", "Delayed")).cast("int"))
    .withColumn("overrun_pct",
        F.round((F.col("actual_spend_gbp") - F.col("budget_gbp")) / F.col("budget_gbp") * 100, 1))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_seng.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_engagements")
print(f"silver_engagements: {df_seng.count():,}")

In [ ]:
# ── silver_timesheets ────────────────────────────────────────────────────────
df_ts = spark.table("bronze_timesheets")

# Total hours per consultant per week
weekly_total = (
    df_ts
    .groupBy("consultant_id", "week_starting")
    .agg(F.sum("hours_logged").alias("total_week_hours"))
)

df_sts = (
    df_ts
    .join(weekly_total, on=["consultant_id", "week_starting"], how="left")
    .withColumn("billable_utilisation_rate",
        F.when(F.col("is_billable") == 1,
            F.round(F.col("hours_logged") / F.col("total_week_hours") * 100, 1)
        ).otherwise(0.0))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sts.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_timesheets")
print(f"silver_timesheets: {df_sts.count():,}")

In [ ]:
print("\n=== Silver Transform Complete ===")
for tbl in ["silver_consultants", "silver_clients", "silver_engagements", "silver_timesheets"]:
    cnt = spark.table(tbl).count()
    print(f"  {tbl}: {cnt:,}")